In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Dense
import xgboost as xgb
from sklearn.metrics import mean_squared_error, precision_score, recall_score
import warnings

warnings.filterwarnings('ignore')

def hybrid_recsys_load_data(file_path):
    data = pd.read_csv(file_path)
    return data

def hybrid_recsys_preprocess_data(data):
    data['Brand_Name'] = data['Brand_Name'].fillna('')
    user_encoder = LabelEncoder()
    item_encoder = LabelEncoder()
    data['user_id'] = user_encoder.fit_transform(data['Client_ID'])
    data['item_id'] = item_encoder.fit_transform(data['Product_ID'])
    return data, user_encoder, item_encoder

def hybrid_recsys_aggregate_data(data):
    data_aggregated = data.groupby(['user_id', 'item_id'], as_index=False)['Monetary_Value'].sum()
    return data_aggregated

def hybrid_recsys_create_interaction_matrix(data_aggregated):
    interaction_matrix = data_aggregated.pivot(index='user_id', columns='item_id', values='Monetary_Value').fillna(0)
    return interaction_matrix

def hybrid_recsys_train_test_split(interaction_matrix):
    train_data, test_data = train_test_split(interaction_matrix, test_size=0.2, random_state=42)
    return train_data, test_data

def hybrid_recsys_train_autoencoder(train_data, test_data, input_dim, encoding_dim=10):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(encoding_dim, activation='relu')(input_layer)
    decoded = Dense(input_dim, activation='sigmoid')(encoded)
    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    autoencoder.fit(train_data, train_data, epochs=10, batch_size=128, shuffle=True, validation_data=(test_data, test_data), verbose=1)
    encoder = Model(input_layer, encoded)
    encoded_train_data = encoder.predict(train_data)
    return encoded_train_data, encoder

def hybrid_recsys_train_xgboost(encoded_train_data, train_data):
    x_train = encoded_train_data
    y_train = train_data.values
    if y_train.ndim == 1:
        y_train = y_train.reshape(-1, 1)
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=10, max_depth=5, learning_rate=0.1)
    model.fit(x_train, y_train)
    return model

def hybrid_recsys_generate_recommendations(user_name, user_encoder, encoded_train_data, model, item_encoder, interaction_matrix, data):
    user_id = user_encoder.transform([user_name])[0]
    user_vector = encoded_train_data[user_id].reshape(1, -1)
    predicted_ratings = model.predict(user_vector).flatten()
    assert len(predicted_ratings) == interaction_matrix.shape[1], "Predicted ratings length mismatch."
    recommendations = pd.DataFrame({'item_id': range(len(predicted_ratings)), 'predicted_rating': predicted_ratings})
    top_10_recommendations = recommendations.sort_values(by='predicted_rating', ascending=False).head(10)
    top_10_recommendations['Product_ID'] = item_encoder.inverse_transform(top_10_recommendations['item_id'])
    top_10_recommendations = top_10_recommendations.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID')
    return top_10_recommendations

def hybrid_recsys_evaluate_recommendations(user_name, user_encoder, item_encoder, interaction_matrix, model, data, k=10):
    user_id = user_encoder.transform([user_name])[0]
    true_ratings = interaction_matrix.loc[user_id].values
    user_vector = encoded_train_data[user_id].reshape(1, -1)
    predicted_ratings = model.predict(user_vector).flatten()
    
    true_top_k = np.argsort(true_ratings)[-k:]
    pred_top_k = np.argsort(predicted_ratings)[-k:]

    true_positives = np.intersect1d(true_top_k, pred_top_k)
    recall = len(true_positives) / len(true_top_k)
    precision = len(true_positives) / len(pred_top_k)
    mse = mean_squared_error(true_ratings, predicted_ratings)
    
    return recall, precision, mse

def main():
    file_path = "/Users/igormol/Desktop/Analytics_Fusion/sample/Marajo.csv"
    data = hybrid_recsys_load_data(file_path)
    data, user_encoder, item_encoder = hybrid_recsys_preprocess_data(data)
    data_aggregated = hybrid_recsys_aggregate_data(data)
    interaction_matrix = hybrid_recsys_create_interaction_matrix(data_aggregated)
    train_data, test_data = hybrid_recsys_train_test_split(interaction_matrix)
    input_dim = interaction_matrix.shape[1]
    encoded_train_data, encoder = hybrid_recsys_train_autoencoder(train_data, test_data, input_dim)
    model = hybrid_recsys_train_xgboost(encoded_train_data, train_data)
    
    users = interaction_matrix.index.tolist()
    recalls, precisions, mses = [], [], []
    
    for user_id in users:
        user_name = user_encoder.inverse_transform([user_id])[0]
        recall, precision, mse = hybrid_recsys_evaluate_recommendations(user_name, user_encoder, item_encoder, interaction_matrix, model, data)
        recalls.append(recall)
        precisions.append(precision)
        mses.append(mse)
        print(f"User: {user_name}, Recall@10: {recall:.4f}, Precision@10: {precision:.4f}, MSE: {mse:.4f}")
    
    print(f"\nMean Recall@10: {np.mean(recalls):.4f}")
    print(f"Mean Precision@10: {np.mean(precisions):.4f}")
    print(f"Overall MSE: {np.mean(mses):.4f}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Dense
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

def hybrid_recsys_load_data(file_path):
    data = pd.read_csv(file_path)
    return data

def hybrid_recsys_preprocess_data(data):
    data['Brand_Name'] = data['Brand_Name'].fillna('')
    user_encoder = LabelEncoder()
    item_encoder = LabelEncoder()
    data['user_id'] = user_encoder.fit_transform(data['Client_ID'])
    data['item_id'] = item_encoder.fit_transform(data['Product_ID'])
    return data, user_encoder, item_encoder

def hybrid_recsys_aggregate_data(data):
    data_aggregated = data.groupby(['user_id', 'item_id'], as_index=False)['Monetary_Value'].sum()
    return data_aggregated

def hybrid_recsys_create_interaction_matrix(data_aggregated):
    interaction_matrix = data_aggregated.pivot(index='user_id', columns='item_id', values='Monetary_Value').fillna(0)
    return interaction_matrix

def hybrid_recsys_train_test_split(interaction_matrix):
    train_data, test_data = train_test_split(interaction_matrix, test_size=0.2, random_state=42)
    return train_data, test_data

def hybrid_recsys_train_autoencoder(train_data, test_data, input_dim, encoding_dim=10):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(encoding_dim, activation='relu')(input_layer)
    decoded = Dense(input_dim, activation='sigmoid')(encoded)
    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    autoencoder.fit(train_data, train_data, epochs=10, batch_size=128, shuffle=True, validation_data=(test_data, test_data), verbose=1)
    encoder = Model(input_layer, encoded)
    encoded_train_data = encoder.predict(train_data)
    return encoded_train_data, encoder

def hybrid_recsys_train_xgboost(encoded_train_data, train_data):
    x_train = encoded_train_data
    y_train = train_data.values
    if y_train.ndim == 1:
        y_train = y_train.reshape(-1, 1)
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=10, max_depth=5, learning_rate=0.1)
    model.fit(x_train, y_train)
    return model

def hybrid_recsys_generate_recommendations(user_name, user_encoder, encoded_train_data, model, item_encoder, interaction_matrix, data):
    user_id = user_encoder.transform([user_name])[0]
    user_vector = encoded_train_data[user_id].reshape(1, -1)
    predicted_ratings = model.predict(user_vector).flatten()
    assert len(predicted_ratings) == interaction_matrix.shape[1], "Predicted ratings length mismatch."
    recommendations = pd.DataFrame({'item_id': range(len(predicted_ratings)), 'predicted_rating': predicted_ratings})
    top_10_recommendations = recommendations.sort_values(by='predicted_rating', ascending=False).head(10)
    top_10_recommendations['Product_ID'] = item_encoder.inverse_transform(top_10_recommendations['item_id'])
    top_10_recommendations = top_10_recommendations.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID')
    return top_10_recommendations

def hybrid_recsys_evaluate_recommendations(user_name, user_encoder, item_encoder, interaction_matrix, model, data, encoded_train_data, k=10):
    user_id = user_encoder.transform([user_name])[0]
    true_ratings = interaction_matrix.loc[user_id].values
    user_vector = encoded_train_data[user_id].reshape(1, -1)
    predicted_ratings = model.predict(user_vector).flatten()
    
    true_top_k = np.argsort(true_ratings)[-k:]
    pred_top_k = np.argsort(predicted_ratings)[-k:]

    true_positives = np.intersect1d(true_top_k, pred_top_k)
    recall = len(true_positives) / len(true_top_k)
    precision = len(true_positives) / len(pred_top_k)
    mse = mean_squared_error(true_ratings, predicted_ratings)
    
    return recall, precision, mse

def main():
    file_path = "/Users/igormol/Desktop/Analytics_Fusion/sample/Marajo.csv"
    data = hybrid_recsys_load_data(file_path)
    data, user_encoder, item_encoder = hybrid_recsys_preprocess_data(data)
    data_aggregated = hybrid_recsys_aggregate_data(data)
    interaction_matrix = hybrid_recsys_create_interaction_matrix(data_aggregated)
    train_data, test_data = hybrid_recsys_train_test_split(interaction_matrix)
    input_dim = interaction_matrix.shape[1]
    encoded_train_data, encoder = hybrid_recsys_train_autoencoder(train_data, test_data, input_dim)
    model = hybrid_recsys_train_xgboost(encoded_train_data, train_data)
    
    users = interaction_matrix.index.tolist()
    recalls, precisions, mses = []

    for user_id in users:
        user_name = user_encoder.inverse_transform([user_id])[0]
        recall, precision, mse = hybrid_recsys_evaluate_recommendations(user_name, user_encoder, item_encoder, interaction_matrix, model, data, encoded_train_data)
        recalls.append(recall)
        precisions.append(precision)
        mses.append(mse)
        print(f"User: {user_name}, Recall@10: {recall:.4f}, Precision@10: {precision:.4f}, MSE: {mse:.4f}")
    
    print(f"\nMean Recall@10: {np.mean(recalls):.4f}")
    print(f"Mean Precision@10: {np.mean(precisions):.4f}")
    print(f"Overall MSE: {np.mean(mses):.4f}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Dense
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

def hybrid_recsys_load_data(file_path):
    data = pd.read_csv(file_path)
    return data

def hybrid_recsys_preprocess_data(data):
    data['Brand_Name'] = data['Brand_Name'].fillna('')
    user_encoder = LabelEncoder()
    item_encoder = LabelEncoder()
    data['user_id'] = user_encoder.fit_transform(data['Client_ID'])
    data['item_id'] = item_encoder.fit_transform(data['Product_ID'])
    return data, user_encoder, item_encoder

def hybrid_recsys_aggregate_data(data):
    data_aggregated = data.groupby(['user_id', 'item_id'], as_index=False)['Monetary_Value'].sum()
    return data_aggregated

def hybrid_recsys_create_interaction_matrix(data_aggregated):
    interaction_matrix = data_aggregated.pivot(index='user_id', columns='item_id', values='Monetary_Value').fillna(0)
    return interaction_matrix

def hybrid_recsys_train_test_split(interaction_matrix):
    train_data, test_data = train_test_split(interaction_matrix, test_size=0.2, random_state=42)
    return train_data, test_data

def hybrid_recsys_train_autoencoder(train_data, test_data, input_dim, encoding_dim=10):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(encoding_dim, activation='relu')(input_layer)
    decoded = Dense(input_dim, activation='sigmoid')(encoded)
    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    autoencoder.fit(train_data, train_data, epochs=10, batch_size=128, shuffle=True, validation_data=(test_data, test_data), verbose=1)
    encoder = Model(input_layer, encoded)
    encoded_train_data = encoder.predict(train_data)
    return encoded_train_data, encoder

def hybrid_recsys_train_xgboost(encoded_train_data, train_data):
    x_train = encoded_train_data
    y_train = train_data.values
    if y_train.ndim == 1:
        y_train = y_train.reshape(-1, 1)
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=10, max_depth=5, learning_rate=0.1)
    model.fit(x_train, y_train)
    return model

def hybrid_recsys_generate_recommendations(user_name, user_encoder, encoded_train_data, model, item_encoder, interaction_matrix, data):
    user_id = user_encoder.transform([user_name])[0]
    user_vector = encoded_train_data[user_id].reshape(1, -1)
    predicted_ratings = model.predict(user_vector).flatten()
    assert len(predicted_ratings) == interaction_matrix.shape[1], "Predicted ratings length mismatch."
    recommendations = pd.DataFrame({'item_id': range(len(predicted_ratings)), 'predicted_rating': predicted_ratings})
    top_10_recommendations = recommendations.sort_values(by='predicted_rating', ascending=False).head(10)
    top_10_recommendations['Product_ID'] = item_encoder.inverse_transform(top_10_recommendations['item_id'])
    top_10_recommendations = top_10_recommendations.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID')
    return top_10_recommendations

def hybrid_recsys_evaluate_recommendations(user_name, user_encoder, item_encoder, interaction_matrix, model, data, encoded_train_data, k=10):
    user_id = user_encoder.transform([user_name])[0]
    true_ratings = interaction_matrix.loc[user_id].values
    user_vector = encoded_train_data[user_id].reshape(1, -1)
    predicted_ratings = model.predict(user_vector).flatten()
    
    true_top_k = np.argsort(true_ratings)[-k:]
    pred_top_k = np.argsort(predicted_ratings)[-k:]

    true_positives = np.intersect1d(true_top_k, pred_top_k)
    recall = len(true_positives) / len(true_top_k)
    precision = len(true_positives) / len(pred_top_k)
    mse = mean_squared_error(true_ratings, predicted_ratings)
    
    return recall, precision, mse

def main():
    file_path = "/Users/igormol/Desktop/Analytics_Fusion/sample/Marajo.csv"
    data = hybrid_recsys_load_data(file_path)
    data, user_encoder, item_encoder = hybrid_recsys_preprocess_data(data)
    data_aggregated = hybrid_recsys_aggregate_data(data)
    interaction_matrix = hybrid_recsys_create_interaction_matrix(data_aggregated)
    train_data, test_data = hybrid_recsys_train_test_split(interaction_matrix)
    input_dim = interaction_matrix.shape[1]
    encoded_train_data, encoder = hybrid_recsys_train_autoencoder(train_data, test_data, input_dim)
    model = hybrid_recsys_train_xgboost(encoded_train_data, train_data)
    
    users = interaction_matrix.index.tolist()
    recalls, precisions, mses = [], [], []
    
    for user_id in users:
        user_name = user_encoder.inverse_transform([user_id])[0]
        recall, precision, mse = hybrid_recsys_evaluate_recommendations(user_name, user_encoder, item_encoder, interaction_matrix, model, data, encoded_train_data)
        recalls.append(recall)
        precisions.append(precision)
        mses.append(mse)
        print(f"User: {user_name}, Recall@10: {recall:.4f}, Precision@10: {precision:.4f}, MSE: {mse:.4f}")
    
    print(f"\nMean Recall@10: {np.mean(recalls):.4f}")
    print(f"Mean Precision@10: {np.mean(precisions):.4f}")
    print(f"Overall MSE: {np.mean(mses):.4f}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error
from keras.models import Model, Sequential
from keras.layers import Dense, Input
import matplotlib.pyplot as plt
import seaborn as sns
import squarify

# Load the data
file_path = '/Users/igormol/Desktop/Analytics_Fusion/sample/Marajo.csv'
data = pd.read_csv(file_path, parse_dates=['Sales_Date'])

# Preprocess data
data['Sales_Date'] = pd.to_datetime(data['Sales_Date'], errors='coerce')
data.dropna(subset=['Sales_Date'], inplace=True)

# Compute RFM scores
latest_date = data['Sales_Date'].max() + pd.DateOffset(1)
rfm_table = data.groupby('Client_ID').agg({
    'Sales_Date': lambda x: (latest_date - x.max()).days,
    'Client_ID': 'count',
    'Monetary_Value': 'sum'
}).reset_index()
rfm_table.columns = ['Client_ID', 'Recency', 'Frequency', 'Monetary']

# Compute Churn index (hypothetical: 1 if no purchase in the last 90 days, else 0)
rfm_table['Churn'] = rfm_table['Recency'].apply(lambda x: 1 if x > 90 else 0)

# Normalize RFM and churn data
scaler = MinMaxScaler()
rfm_normalized = scaler.fit_transform(rfm_table[['Recency', 'Frequency', 'Monetary', 'Churn']])

# Build and train an autoencoder
input_dim = rfm_normalized.shape[1]
encoding_dim = 2

input_layer = Input(shape=(input_dim,))
encoder = Dense(encoding_dim, activation="tanh")(input_layer)
decoder = Dense(input_dim, activation="linear")(encoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Train the autoencoder
autoencoder.fit(rfm_normalized, rfm_normalized, epochs=50, batch_size=8, shuffle=True, validation_split=0.2)

# Encode the data
encoder_model = Model(inputs=input_layer, outputs=encoder)
encoded_data = encoder_model.predict(rfm_normalized)

# Clustering
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(encoded_data)

# Add cluster information to the RFM table
rfm_table['Cluster'] = clusters

# Compute MAE for clustering evaluation
decoded_data = autoencoder.predict(rfm_normalized)
mae = mean_absolute_error(rfm_normalized, decoded_data)
print(f"Mean Absolute Error: {mae}")

# Visualize the clusters using a treemap
cluster_summary = rfm_table.groupby('Cluster').mean().reset_index()
cluster_summary['Count'] = rfm_table['Cluster'].value_counts().values
cluster_summary['Labels'] = ['Cluster ' + str(i) for i in cluster_summary['Cluster']]

plt.figure(figsize=(12, 8))
squarify.plot(sizes=cluster_summary['Count'], label=cluster_summary['Labels'], alpha=.8)
plt.axis('off')
plt.title("Treemap of Clusters")
plt.show()

# Plot mean RFM and churn index for each cluster
cluster_summary.set_index('Labels')[['Recency', 'Frequency', 'Monetary', 'Churn']].plot(kind='bar', figsize=(12, 8))
plt.title("Mean RFM and Churn Index for Each Cluster")
plt.ylabel("Mean Value")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error
from keras.models import Model
from keras.layers import Dense, Input
import matplotlib.pyplot as plt
import squarify

# Load the data
file_path = '/Users/igormol/Desktop/Analytics_Fusion/sample/Marajo.csv'
data = pd.read_csv(file_path, parse_dates=['Sales_Date'])

# Preprocess data
data['Sales_Date'] = pd.to_datetime(data['Sales_Date'], errors='coerce')
data.dropna(subset=['Sales_Date'], inplace=True)

# Compute RFM scores
latest_date = data['Sales_Date'].max() + pd.DateOffset(1)
rfm_table = data.groupby('Client_ID').agg({
    'Sales_Date': lambda x: (latest_date - x.max()).days,
    'Client_ID': 'count',
    'Monetary_Value': 'sum'
}).reset_index()
rfm_table.columns = ['Client_ID', 'Recency', 'Frequency', 'Monetary']

# Compute Churn index (hypothetical: 1 if no purchase in the last 90 days, else 0)
rfm_table['Churn'] = rfm_table['Recency'].apply(lambda x: 1 if x > 90 else 0)

# Normalize RFM and churn data
scaler = MinMaxScaler()
rfm_normalized = scaler.fit_transform(rfm_table[['Recency', 'Frequency', 'Monetary', 'Churn']])

# Build and train an autoencoder
input_dim = rfm_normalized.shape[1]
encoding_dim = 2

input_layer = Input(shape=(input_dim,))
encoder = Dense(encoding_dim, activation="tanh")(input_layer)
decoder = Dense(input_dim, activation="linear")(encoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Train the autoencoder
autoencoder.fit(rfm_normalized, rfm_normalized, epochs=50, batch_size=8, shuffle=True, validation_split=0.2)

# Encode the data
encoder_model = Model(inputs=input_layer, outputs=encoder)
encoded_data = encoder_model.predict(rfm_normalized)

# Clustering
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(encoded_data)

# Add cluster information to the RFM table
rfm_table['Cluster'] = clusters

# Compute MAE for clustering evaluation
decoded_data = autoencoder.predict(rfm_normalized)
mae = mean_absolute_error(rfm_normalized, decoded_data)
print(f"Mean Absolute Error: {mae}")

# Visualize the clusters using a treemap
cluster_summary = rfm_table.groupby('Cluster').mean().reset_index()
cluster_summary['Count'] = rfm_table['Cluster'].value_counts().values
cluster_summary['Labels'] = ['Cluster ' + str(i) for i in cluster_summary['Cluster']]

plt.figure(figsize=(12, 8))
squarify.plot(sizes=cluster_summary['Count'], label=cluster_summary['Labels'], alpha=.8)
plt.axis('off')
plt.title("Treemap of Clusters")
plt.show()

# Plot mean RFM and churn index for each cluster
cluster_summary.set_index('Labels')[['Recency', 'Frequency', 'Monetary', 'Churn']].plot(kind='bar', figsize=(12, 8))
plt.title("Mean RFM and Churn Index for Each Cluster")
plt.ylabel("Mean Value")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error
from keras.models import Model
from keras.layers import Dense, Input
import matplotlib.pyplot as plt
import squarify

# Load the data
file_path = '/Users/igormol/Desktop/Analytics_Fusion/sample/Marajo.csv'
data = pd.read_csv(file_path, parse_dates=['Sales_Date'])

# Preprocess data
data['Sales_Date'] = pd.to_datetime(data['Sales_Date'], errors='coerce')
data.dropna(subset=['Sales_Date'], inplace=True)

# Compute RFM scores
latest_date = data['Sales_Date'].max() + pd.DateOffset(1)
rfm_table = data.groupby('Client_ID').agg({
    'Sales_Date': lambda x: (latest_date - x.max()).days,
    'Client_ID': 'count',
    'Monetary_Value': 'sum'
}).rename(columns={'Client_ID': 'Frequency'}).reset_index()
rfm_table.columns = ['Client_ID', 'Recency', 'Frequency', 'Monetary']

# Compute Churn index (hypothetical: 1 if no purchase in the last 90 days, else 0)
rfm_table['Churn'] = rfm_table['Recency'].apply(lambda x: 1 if x > 90 else 0)

# Normalize RFM and churn data
scaler = MinMaxScaler()
rfm_normalized = scaler.fit_transform(rfm_table[['Recency', 'Frequency', 'Monetary', 'Churn']])

# Build and train an autoencoder
input_dim = rfm_normalized.shape[1]
encoding_dim = 2

input_layer = Input(shape=(input_dim,))
encoder = Dense(encoding_dim, activation="tanh")(input_layer)
decoder = Dense(input_dim, activation="linear")(encoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Train the autoencoder
autoencoder.fit(rfm_normalized, rfm_normalized, epochs=50, batch_size=8, shuffle=True, validation_split=0.2)

# Encode the data
encoder_model = Model(inputs=input_layer, outputs=encoder)
encoded_data = encoder_model.predict(rfm_normalized)

# Clustering
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(encoded_data)

# Add cluster information to the RFM table
rfm_table['Cluster'] = clusters

# Compute MAE for clustering evaluation
decoded_data = autoencoder.predict(rfm_normalized)
mae = mean_absolute_error(rfm_normalized, decoded_data)
print(f"Mean Absolute Error: {mae}")

# Visualize the clusters using a treemap
cluster_summary = rfm_table.groupby('Cluster').mean().reset_index()
cluster_summary['Count'] = rfm_table['Cluster'].value_counts().values
cluster_summary['Labels'] = ['Cluster ' + str(i) for i in cluster_summary['Cluster']]

plt.figure(figsize=(12, 8))
squarify.plot(sizes=cluster_summary['Count'], label=cluster_summary['Labels'], alpha=.8)
plt.axis('off')
plt.title("Treemap of Clusters")
plt.show()

# Plot mean RFM and churn index for each cluster
cluster_summary.set_index('Labels')[['Recency', 'Frequency', 'Monetary', 'Churn']].plot(kind='bar', figsize=(12, 8))
plt.title("Mean RFM and Churn Index for Each Cluster")
plt.ylabel("Mean Value")
plt.show()
